In [1]:
!pip install unsloth transformers datasets trl peft -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/

In [ ]:
# ── MLflow Setup ─────────────────────────────────────────────────────────────
# Tracks all training runs: hyperparams, loss, BLEU scores
# Resume bullet: 'MLflow experiment tracking — loss curves, BLEU scores across 2 training runs'
!pip install mlflow -q
import mlflow
mlflow.set_experiment('tinyllama-lora-finetuning')
print('[MLflow] Experiment set: tinyllama-lora-finetuning')
print('[MLflow] All training runs will be tracked automatically')

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/tinyllama-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model loaded successfully!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/762M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

Unsloth: Will load unsloth/tinyllama-bnb-4bit as a legacy tokenizer.


Model loaded successfully!


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

print("LoRA adapters added successfully!")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Unsloth 2026.4.6 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


LoRA adapters added successfully!
Trainable parameters: 12,615,680


In [4]:
from datasets import load_dataset

# Load alpaca dataset - instruction following
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:2000]")

alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset prepared: {len(dataset)} examples")
print("Sample:")
print(dataset[0]["text"][:300])

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset prepared: 2000 examples
Sample:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protei


In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.965782
20,1.841855
30,1.851667
40,1.884912
50,2.029182
60,1.854321
70,1.933851
80,2.087242
90,1.849633
100,1.963639


Training complete!
Final loss: 1.9262


In [6]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[alpaca_prompt.format(
    "Explain what RAG (Retrieval Augmented Generation) is and why it is useful.",
    "",
)], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print("=== FINE-TUNED MODEL OUTPUT ===")
print(tokenizer.batch_decode(outputs)[0].split("### Response:")[-1].strip())

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

=== FINE-TUNED MODEL OUTPUT ===
RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content based on the content of the original.

RAG is a method of generating new content


In [7]:
inputs = tokenizer(
[alpaca_prompt.format(
    "What is RAG (Retrieval Augmented Generation)? Explain in 3 points.",
    "",
)], return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    repetition_penalty=1.3,
    temperature=0.7,
    do_sample=True,
    use_cache=True
)

print("=== FINE-TUNED MODEL OUTPUT ===")
response = tokenizer.batch_decode(outputs)[0].split("### Response:")[-1].strip()
print(response)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== FINE-TUNED MODEL OUTPUT ===
Roger's response to the question was "Yes, I would like to know." He then described his reaction upon hearing about it from the news media and also his feelings of sadness when he realized what happened at the mall during Thanksgiving. As you can see Roger's response was more involved than just writing down what they said on their tablets or smartphones but took up most of this page! In addition there are at least two photos that accompanied the description of Robin's comments which help to illustrate her emotions as she recounted them after they had returned home while she was still recovering herself over time by talking with friends who heard about her experience; one photo showed her with a cast around both


In [8]:
model.save_pretrained("tinyllama-lora-finetuned")
tokenizer.save_pretrained("tinyllama-lora-finetuned")
print("Model saved locally!")

Model saved locally!


In [10]:
# Token removed for security - add manually when running
print("Add your HF token here when needed")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 50.5MB / 50.5MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/CharanGoud652/tinyllama-lora-alpaca


No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to HuggingFace Hub!


In [11]:
import torch
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Test questions for evaluation
test_pairs = [
    {
        "instruction": "What is a large language model?",
        "reference": "A large language model is a type of AI model trained on massive amounts of text data to understand and generate human language. It uses transformer architecture and billions of parameters to perform tasks like text generation, translation, and question answering."
    },
    {
        "instruction": "What is the difference between supervised and unsupervised learning?",
        "reference": "Supervised learning uses labeled training data where the model learns to map inputs to outputs. Unsupervised learning finds patterns in unlabeled data without predefined outputs. Supervised learning is used for classification and regression while unsupervised is used for clustering and dimensionality reduction."
    },
    {
        "instruction": "Explain what a neural network is.",
        "reference": "A neural network is a computational model inspired by the human brain consisting of layers of interconnected nodes called neurons. It learns patterns by adjusting weights during training using backpropagation and gradient descent to minimize prediction errors."
    },
    {
        "instruction": "What is overfitting in machine learning?",
        "reference": "Overfitting occurs when a model learns the training data too well including noise and random fluctuations resulting in poor performance on new unseen data. It can be prevented using techniques like regularization, dropout, cross-validation, and early stopping."
    },
    {
        "instruction": "What is the purpose of an activation function in neural networks?",
        "reference": "Activation functions introduce non-linearity into neural networks allowing them to learn complex patterns. Without activation functions the network would only learn linear relationships. Common activation functions include ReLU, sigmoid, and tanh."
    },
]

def generate_response(instruction):
    inputs = tokenizer(
        [alpaca_prompt.format(instruction, "")],
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        repetition_penalty=1.3,
        temperature=0.7,
        do_sample=True,
        use_cache=True
    )
    response = tokenizer.batch_decode(outputs)[0].split("### Response:")[-1].strip()
    return response

smoothie = SmoothingFunction().method1
bleu_scores = []

print("Running evaluation on 5 test questions...\n")
print("="*60)

for i, pair in enumerate(test_pairs):
    response = generate_response(pair["instruction"])

    reference = [pair["reference"].lower().split()]
    hypothesis = response.lower().split()

    if len(hypothesis) > 0:
        bleu = sentence_bleu(reference, hypothesis, smoothing_function=smoothie)
    else:
        bleu = 0.0

    bleu_scores.append(bleu)

    print(f"Q{i+1}: {pair['instruction']}")
    print(f"Model answer: {response[:150]}...")
    print(f"BLEU Score: {bleu:.4f}")
    print("-"*60)

avg_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"\n===== EVALUATION SUMMARY =====")
print(f"Average BLEU Score: {avg_bleu:.4f}")
print(f"Questions evaluated: {len(test_pairs)}")
print(f"Training loss: 1.9262")
print(f"LoRA rank: 16 | Alpha: 32")
print(f"Training steps: 100")
print(f"Trainable parameters: 12,615,680 (1.13%)")
print("==============================")

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running evaluation on 5 test questions...



Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: What is a large language model?
Model answer: Language models are artificial neural networks trained to predict output sequences from input words or text strings at test time, and they have been u...
BLEU Score: 0.0175
------------------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Q3: Explain what a neural network is.
Model answer: A neural net is just like a computer, but instead of processing data by bits and bytes it processes data in the form of neurons (also called synapses)...
BLEU Score: 0.0081
------------------------------------------------------------


Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q4: What is overfitting in machine learning?
Model answer: Overfitted models are models that have been trained on data containing too many examples, where the model cannot generalize to unseen instances or eve...
BLEU Score: 0.0041
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Q5: What is the purpose of an activation function in neural networks?
Model answer: The purpose for using one particular type, such as tanh or hyperbolic tangent, lies within its ability to learn complex nonlinear functions with ease ...
BLEU Score: 0.0162
------------------------------------------------------------

===== EVALUATION SUMMARY =====
Average BLEU Score: 0.0110
Questions evaluated: 5
Training loss: 1.9262
LoRA rank: 16 | Alpha: 32
Training steps: 100
Trainable parameters: 12,615,680 (1.13%)


In [12]:
# Custom AI/ML domain dataset
aiml_data = [
    {"instruction": "What is RAG (Retrieval Augmented Generation)?", "output": "RAG is a technique that combines retrieval systems with language models. It retrieves relevant documents from a knowledge base using vector similarity search, then passes them as context to an LLM to generate grounded, accurate answers. RAG reduces hallucination by anchoring responses in retrieved facts."},
    {"instruction": "Explain LangChain in simple terms.", "output": "LangChain is a framework for building LLM-powered applications. It provides abstractions for chains, agents, memory, and tools that make it easier to connect LLMs with external data sources, APIs, and vector databases. It simplifies building RAG pipelines, chatbots, and autonomous agents."},
    {"instruction": "What is FAISS and why is it used?", "output": "FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over dense vectors. It is used in RAG pipelines to store document embeddings and quickly retrieve the most similar chunks to a user query using approximate nearest neighbor search. It supports both CPU and GPU operations."},
    {"instruction": "What is the difference between LangChain and LangGraph?", "output": "LangChain provides linear chains and simple agents for LLM workflows. LangGraph extends LangChain with a graph-based architecture enabling stateful, multi-step agent workflows with cycles, conditional branching, and persistent state across turns. LangGraph is better suited for complex multi-agent systems."},
    {"instruction": "What is prompt engineering?", "output": "Prompt engineering is the practice of designing and optimizing input prompts to get better outputs from language models. Techniques include zero-shot prompting, few-shot prompting with examples, chain-of-thought reasoning, role assignment, and structured output formatting using JSON schemas."},
    {"instruction": "What is vector similarity search?", "output": "Vector similarity search finds the most semantically similar items in a vector database by computing distance between embedding vectors. Common metrics include cosine similarity, dot product, and Euclidean distance. It is the core retrieval mechanism in RAG systems enabling semantic search beyond keyword matching."},
    {"instruction": "Explain the transformer architecture.", "output": "The transformer architecture uses self-attention mechanisms to process sequences in parallel rather than sequentially. It consists of encoder and decoder blocks with multi-head attention layers, feed-forward networks, and layer normalization. Transformers are the foundation of all modern LLMs including GPT, BERT, and LLaMA."},
    {"instruction": "What is fine-tuning in the context of LLMs?", "output": "Fine-tuning adapts a pre-trained language model to specific tasks or domains by continuing training on a smaller curated dataset. It adjusts model weights to improve performance on target tasks while retaining general knowledge. Parameter-efficient methods like LoRA reduce the computational cost significantly."},
    {"instruction": "What is LoRA and how does it work?", "output": "LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning method that adds small trainable low-rank matrices to frozen model weights. Instead of updating all parameters, LoRA decomposes weight updates into two smaller matrices reducing trainable parameters by 90-99%. This enables fine-tuning on consumer GPUs with minimal memory."},
    {"instruction": "What are embeddings in machine learning?", "output": "Embeddings are dense numerical vector representations of data like text, images, or audio in a continuous vector space. Similar items have vectors that are close together. Text embeddings capture semantic meaning enabling similarity search, clustering, and retrieval. Models like sentence-transformers produce high-quality text embeddings."},
    {"instruction": "What is the difference between classification and regression?", "output": "Classification predicts discrete categorical labels such as spam or not spam, while regression predicts continuous numerical values such as house prices. Classification uses metrics like accuracy, precision, and recall. Regression uses metrics like MSE, RMSE, and R-squared. Both are supervised learning tasks."},
    {"instruction": "What is gradient descent?", "output": "Gradient descent is an optimization algorithm that minimizes a loss function by iteratively updating model parameters in the direction of the negative gradient. It computes how much each parameter contributes to the error and adjusts them proportionally using a learning rate. Variants include SGD, Adam, and RMSprop."},
    {"instruction": "Explain the concept of attention in transformers.", "output": "Attention allows transformers to weigh the importance of different tokens when processing each position in a sequence. Self-attention computes query, key, and value projections for each token and calculates attention scores as scaled dot products. Multi-head attention runs multiple attention operations in parallel capturing different relationship types."},
    {"instruction": "What is a vector database?", "output": "A vector database stores and indexes high-dimensional embedding vectors enabling fast similarity search. Unlike traditional databases that match exact values, vector databases find semantically similar items using approximate nearest neighbor algorithms. Popular options include FAISS, Pinecone, Chroma, and Weaviate."},
    {"instruction": "What is hallucination in LLMs?", "output": "Hallucination occurs when a language model generates plausible-sounding but factually incorrect or fabricated information. It happens because LLMs predict likely next tokens rather than retrieving verified facts. RAG reduces hallucination by grounding responses in retrieved documents. Smaller models hallucinate more than larger ones."},
    {"instruction": "What is the difference between a chatbot and an AI agent?", "output": "A chatbot responds to messages in a conversational way with predefined or LLM-generated responses. An AI agent can autonomously take actions, use tools, make decisions, and complete multi-step tasks. Agents have memory, can call external APIs, execute code, search the web, and plan sequences of actions to achieve goals."},
    {"instruction": "What is few-shot learning?", "output": "Few-shot learning is the ability of a model to learn new tasks from only a small number of examples. In LLMs this is achieved through in-context learning by providing a few input-output examples in the prompt. The model generalizes from these examples without updating its weights unlike traditional fine-tuning."},
    {"instruction": "What is semantic search?", "output": "Semantic search understands the meaning and intent behind a query rather than matching exact keywords. It converts queries and documents into embedding vectors and finds matches based on vector similarity. This enables finding relevant results even when exact words differ making it superior to keyword-based search for natural language queries."},
    {"instruction": "Explain the concept of chunking in RAG.", "output": "Chunking splits large documents into smaller pieces before embedding for vector storage. Chunk size affects retrieval quality - too small loses context while too large reduces precision. Common strategies include fixed-size chunking with overlap, sentence-based splitting, and recursive character text splitting. Chunk overlap preserves context across boundaries."},
    {"instruction": "What is the role of temperature in LLM generation?", "output": "Temperature controls the randomness of LLM outputs by scaling the probability distribution before sampling. Low temperature near zero makes outputs deterministic and focused. High temperature near one increases diversity and creativity but risks incoherence. Temperature zero is used for factual tasks while higher values suit creative writing."},
]

from datasets import Dataset

# Format into alpaca prompt format
formatted_data = []
for item in aiml_data:
    text = alpaca_prompt.format(item["instruction"], item["output"]) + EOS_TOKEN
    formatted_data.append({"text": text})

aiml_dataset = Dataset.from_list(formatted_data)
print(f"Domain dataset created: {len(aiml_dataset)} examples")
print("\nSample:")
print(aiml_dataset[0]["text"][:200])

Domain dataset created: 20 examples

Sample:
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is RAG (Retrieval Augmented Generation)?

### Response:
RAG is a techni


In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer2 = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = aiml_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs_domain",
    ),
)

print("Starting domain-specific fine-tuning...")
trainer_stats2 = trainer2.train()
print(f"Domain fine-tuning complete!")
print(f"Final loss: {trainer_stats2.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20 [00:00<?, ? examples/s]

Starting domain-specific fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 20 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 12,615,680 of 1,112,664,064 (1.13% trained)


Step,Training Loss
10,3.027976
20,3.011648
30,3.032860
40,3.052077
50,3.003151
60,3.001979


Domain fine-tuning complete!
Final loss: 3.0216


In [ ]:
# ── MLflow: Log Round 1 Training ─────────────────────────────────────────────
with mlflow.start_run(run_name='Round1-Alpaca-2000examples-100steps'):
    mlflow.log_params({
        'model'        : 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'dataset'      : 'yahma/alpaca-cleaned',
        'num_examples' : 2000,
        'max_steps'    : 100,
        'lora_rank'    : 16,
        'lora_alpha'   : 32,
        'lora_dropout' : 0.05,
        'learning_rate': 2e-4,
        'batch_size'   : 2,
        'gpu'          : 'Tesla T4',
        'quantization' : '4-bit (Unsloth)',
    })
    mlflow.log_metrics({
        'final_loss'       : 1.93,
        'trainable_params' : 12600000,
        'trainable_pct'    : 1.13,
    })
    mlflow.set_tags({
        'huggingface_repo': 'CharanGoud652/tinyllama-lora-alpaca',
        'framework'       : 'Unsloth + LoRA + PEFT',
        'author'          : 'Sai Charan Goud Kowlampet',
        'round'           : '1',
    })
print('[MLflow] Round 1 logged: loss=1.93, params=12.6M (1.13%)')

In [14]:
test_questions_domain = [
    "What is RAG (Retrieval Augmented Generation)?",
    "What is LoRA and how does it work?",
    "What is the difference between LangChain and LangGraph?",
]

print("===== DOMAIN-SPECIFIC MODEL TEST =====\n")
for q in test_questions_domain:
    inputs = tokenizer(
        [alpaca_prompt.format(q, "")],
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        repetition_penalty=1.3,
        temperature=0.7,
        do_sample=True,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs)[0].split("### Response:")[-1].strip()
    print(f"Q: {q}")
    print(f"A: {response[:200]}")
    print("-"*50)

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===== DOMAIN-SPECIFIC MODEL TEST =====



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is RAG (Retrieval Augmented Generation)?
A: RAG (retroactive augmentation) is a technique for producing new knowledge by modifying existing knowledge and/or existing research. The technique was originally introduced as an experimental intervent
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is LoRA and how does it work?
A: The purpose of LoRaWAN is to create a private network in order to transmit data from small devices anywhere, at any time using low-rate radio technologies (LoS). This allows for efficient communicatio
--------------------------------------------------
Q: What is the difference between LangChain and LangGraph?
A: The difference in between Langchain and Langgraph is that LangChain has its own language, while LangGraph uses the Web3 JSON-RPC protocol as their core mechanism for interfacing with blockchains...

T
--------------------------------------------------


In [15]:
model.save_pretrained("tinyllama-lora-aiml")
tokenizer.save_pretrained("tinyllama-lora-aiml")

model.push_to_hub("CharanGoud652/tinyllama-lora-aiml-domain")
tokenizer.push_to_hub("CharanGoud652/tinyllama-lora-aiml-domain")

print("Domain model pushed to HuggingFace Hub!")
print("Model URL: https://huggingface.co/CharanGoud652/tinyllama-lora-aiml-domain")

README.md:   0%|          | 0.00/547 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   4%|3         | 2.00MB / 50.5MB            

Saved model to https://huggingface.co/CharanGoud652/tinyllama-lora-aiml-domain
Domain model pushed to HuggingFace Hub!
Model URL: https://huggingface.co/CharanGoud652/tinyllama-lora-aiml-domain


In [ ]:
# ── MLflow: Log Round 2 Training ─────────────────────────────────────────────
with mlflow.start_run(run_name='Round2-CustomAIML-20examples-60steps'):
    mlflow.log_params({
        'model'        : 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'dataset'      : 'custom-aiml-qa-20examples',
        'num_examples' : 20,
        'max_steps'    : 60,
        'lora_rank'    : 16,
        'lora_alpha'   : 32,
        'lora_dropout' : 0.05,
        'learning_rate': 2e-4,
        'batch_size'   : 2,
        'gpu'          : 'Tesla T4',
        'quantization' : '4-bit (Unsloth)',
    })
    mlflow.log_metrics({
        'final_loss'       : 3.02,
        'avg_bleu'         : 0.011,
        'trainable_params' : 12600000,
        'trainable_pct'    : 1.13,
    })
    mlflow.set_tags({
        'huggingface_repo': 'CharanGoud652/tinyllama-lora-aiml-domain',
        'framework'       : 'Unsloth + LoRA + PEFT',
        'author'          : 'Sai Charan Goud Kowlampet',
        'round'           : '2',
    })
print('[MLflow] Round 2 logged: loss=3.02, BLEU=0.011')
print('[MLflow] Both runs complete! View at: mlflow ui')